In [ ]:
#@title Estilo de la clase (ejecutar, no hace falta leer) {display-mode: "form"}
from IPython.display import HTML, display

display(HTML(r"""
<style>
@import url('https://fonts.googleapis.com/css2?family=Work+Sans:wght@400;600&family=Amiri:wght@400;700&display=swap');
.rendered_html, .markdown, .cell .text_cell_render { font-family:'Work Sans',system-ui,sans-serif; color:#122535; }
.rendered_html h1,.rendered_html h2,.rendered_html h3 { font-family:'Amiri',Georgia,serif; color:#00529B; }
.rendered_html h2 { border-bottom:2px solid #00529B; padding-bottom:.2em; }
.rendered_html a { color:#00529B; }
.rendered_html table th { background:#00529B; color:#fff; }
.caja { background:#f3f5f7; border-left:4px solid #00529B; padding:.7em 1em; border-radius:4px; }
.ojo { background:#fdf3ec; border-left:4px solid #C8651B; padding:.7em 1em; border-radius:4px; }
</style>
"""))

# Clase 9 · Reducción de dimensionalidad (PCA)

**Analítica de Datos** · Maestría en Ciencias del Comportamiento · Universidad de San Andrés

**Primavera 2026 · 03/10/2026**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomdamelio/analitica_de_datos_alumnos/blob/main/clases/clase-09/notebooks/clase09_python.ipynb)

---

Hasta la clase pasada siempre había una columna para predecir. Hoy no: tenemos **once medidas de
desempeño** de cada empleado de Nimbus y queremos resumirlas en pocas, como se hizo con el Big Five.

Al terminar esta notebook vas a poder:

- encontrar las **componentes principales** con `PCA` de scikit-learn,
- leer **loadings**, **scores** y el **porcentaje de varianza explicada**,
- decidir **cuántas componentes** quedarte con un scree plot,
- **escalar** antes de PCA y explicar por qué,
- usar PCA adentro de un **`Pipeline`**, sin leakage, para predecir la renuncia.

## Contenido

| # | Tema | La idea en una línea |
|---|---|---|
| 1 | [Once medidas de desempeño](#sec-datos) | muchas dicen casi lo mismo |
| 2 | [PCA con dos medidas](#sec-2d) | centrar, encontrar PC1, leer loadings y scores |
| 3 | [Las once a la vez](#sec-once) | dos componentes con nombre, scree y biplot |
| 4 | [¿Y si no escalamos?](#sec-escalar) | gana la medida con la unidad más grande |
| 5 | [PC1 y la renuncia](#sec-renuncia) | una columna predice como once |
| 6 | [Leakage: el experimento del ruido](#sec-leakage) | preprocesar con todos los datos engaña |
| 7 | [¿Cuántas componentes?](#sec-cuantas) | si el fin es predecir, lo decide la validación cruzada |
| 8 | [Más allá de lo lineal](#sec-nolineal) | t-SNE ve lo que PCA no ve |
| 9 | [Ejercicio de cierre: la encuesta de clima](#sec-clima) | sin redundancia, PCA no comprime |
| 10 | [Cierre](#sec-cierre) | lo que te llevás |

## 1. Once medidas de desempeño

<a id="sec-datos"></a>

Recursos humanos de Nimbus junta, para cada uno de los 600 empleados, once medidas de desempeño
que salen de sistemas distintos: el tablero de objetivos, el gestor de tareas, la evaluación del
líder, la de los compañeros. Están en `nimbus_desempeno.csv`. La renuncia está en otra tabla,
`nimbus_rrhh.csv`; la vamos a usar recién en la sección 5, pero la unimos desde ahora por
`empleado_id`.

Nimbus es una empresa **simulada** para esta materia: los datos se generaron con una estructura
conocida, así que al final podemos chequear si PCA la encuentra.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 49

# Colores de la clase: PC1 en verde y PC2 en violeta, como en las slides.
VERDE = "#31A354"
VIOLETA = "#756BB1"
AZUL = "#00529B"
GRAFITO = "#4A5B6B"

# Carpeta de figuras de la página de la clase. En Colab no existe, y la figura no se guarda.
CARPETA_FIGURAS = "../assets/figures"

BASE = "https://raw.githubusercontent.com/tomdamelio/analitica_de_datos_alumnos/main/data/toy-nimbus/"

desempeno = pd.read_csv(BASE + "nimbus_desempeno.csv")
rrhh = pd.read_csv(BASE + "nimbus_rrhh.csv")

datos = desempeno.merge(rrhh[["empleado_id", "renuncia"]], on="empleado_id")

MEDIDAS = [
    "objetivos_cumplidos_pct",
    "entregas_a_tiempo_pct",
    "tareas_cerradas_mes",
    "horas_foco_semana",
    "evaluacion_lider",
    "okr_score",
    "retrabajos_mes",
    "evaluacion_pares",
    "minutos_mentoria_mes",
    "revisiones_a_otros_mes",
    "iniciativas_internas_anio",
]

X = datos[MEDIDAS]

print(X.shape[0], "empleados y", X.shape[1], "medidas")
X.head()

Mirá el promedio y el desvío de cada medida. Están en **unidades muy distintas**: porcentajes,
conteos por mes, notas de 1 a 5, un puntaje de 0 a 1, minutos. Eso va a importar en la sección 4.

In [ ]:
X.describe().T[["mean", "std", "min", "max"]].round(2)

¿Cuánto se repiten entre sí? La **matriz de correlaciones** muestra, para cada par de medidas,
cuánto se mueven juntas: cerca de 1, casi lo mismo; cerca de 0, nada que ver. Con once medidas hay
55 pares distintos, demasiados para mirarlos de a uno en gráficos de dispersión.

In [ ]:
correlaciones = X.corr()

fig, ax = plt.subplots(figsize=(7.5, 6.5), dpi=150)
imagen = ax.imshow(correlaciones, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(MEDIDAS)))
ax.set_yticks(range(len(MEDIDAS)))
ax.set_xticklabels(MEDIDAS, rotation=90, fontsize=8)
ax.set_yticklabels(MEDIDAS, fontsize=8)
fig.colorbar(imagen, ax=ax, shrink=0.8, label="correlación")
ax.set_title("Correlación entre las once medidas")
fig.tight_layout()

if os.path.isdir(CARPETA_FIGURAS):
    fig.savefig(CARPETA_FIGURAS + "/clase09_correlaciones.png", bbox_inches="tight")
plt.show()

Se ven **dos bloques**. Las seis primeras medidas (objetivos, entregas, tareas, foco, líder, OKR)
correlacionan fuerte entre sí, y retrabajos correlaciona con ellas pero en negativo. Las cuatro
últimas (pares, mentoría, revisiones, iniciativas) forman otro bloque. Entre un bloque y el otro,
casi nada.

Once columnas, pero parece que hay **dos cosas de fondo**. PCA es la herramienta para encontrarlas
sin mirar el gráfico a ojo.

## 2. PCA con dos medidas

<a id="sec-2d"></a>

Para ver cómo funciona por dentro, empezamos con dos medidas: **objetivos cumplidos** y
**entregas a tiempo**. Las dos están en porcentaje y con desvíos parecidos, así que por ahora
alcanza con **centrar**: restarle a cada medida su promedio. PCA describe cómo varían los datos
alrededor del promedio; después de centrar, el promedio de cada columna queda en cero.

In [ ]:
par = X[["objetivos_cumplidos_pct", "entregas_a_tiempo_pct"]]

medias = par.mean()
centrado = par - medias

print("promedios originales:")
print(medias.round(2))
print()
print("promedios después de centrar:")
print(centrado.mean().round(10))

Ahora `PCA`. Se crea, se ajusta con `.fit()` y se leen dos cosas:

- **`components_`**: una fila por componente. Cada fila tiene un **loading** por medida: cuánto
  pesa esa medida en la componente.
- **`explained_variance_ratio_`**: qué proporción de toda la variación de los datos captura cada
  componente.

### ✏️ Consigna 1

Ajustá un PCA con **dos componentes** sobre los datos **centrados** y mostrá sus loadings y la
proporción de varianza explicada.

Pistas:

- `PCA(n_components=2)` crea el modelo, todavía vacío;
- `.fit()` lo ajusta: pasale la tabla `centrado`, no la original;
- después de ajustar, los loadings están en `.components_` y la varianza en
  `.explained_variance_ratio_`.

In [ ]:
from sklearn.decomposition import PCA

# TODO: dos componentes
pca_par = PCA(n_components=___)
# TODO: ajustalo con los datos centrados
pca_par.fit(___)

loadings_par = pd.DataFrame(pca_par.components_, columns=par.columns, index=["PC1", "PC2"])
print(loadings_par.round(3))
print()
print("proporción de varianza explicada:", pca_par.explained_variance_ratio_.round(3))

**PC1** pesa parecido en las dos medidas, las dos en positivo: es la dirección en la que objetivos
y entregas **crecen juntas**. Captura más del 80 % de la variación de las dos. **PC2** es la
dirección perpendicular: una sube y la otra baja.

Dos propiedades de los loadings de cada componente:

- **al cuadrado suman 1**; si no, PCA podría inflar los pesos y la varianza crecería sin límite;
- **el signo es arbitrario**: otro programa, por ejemplo R, puede devolver los mismos números con
  el signo cambiado. Es la misma recta, recorrida al revés.

Lo verificamos, y de paso calculamos el ángulo de PC1 respecto del eje de objetivos.

In [ ]:
phi = pca_par.components_[0]

print("loadings de PC1:", phi.round(3))
print("suma de sus cuadrados:", round(phi[0] ** 2 + phi[1] ** 2, 6))

angulo = np.degrees(np.arctan2(phi[1], phi[0]))
print("ángulo de PC1:", round(angulo, 1), "grados")

El **score** de un empleado en PC1 es su nota en esa componente: sus desvíos del promedio,
multiplicados por los loadings y sumados. Es exactamente dónde cae su proyección sobre la recta de
PC1. Lo calculamos a mano para un empleado y lo comparamos con lo que devuelve `.transform()`.

### ✏️ Consigna 2

Calculá a mano el score en PC1 del empleado de la fila 13 y comparalo con el que devuelve
`.transform()`.

Pistas:

- el score es cada desvío del promedio multiplicado por su loading, y todo sumado;
- los loadings de PC1 están en `phi` (`phi[0]` es el de objetivos, `phi[1]` el de entregas);
- los desvíos del empleado están en `desvio` (`desvio.iloc[0]` y `desvio.iloc[1]`, en el mismo
  orden).

In [ ]:
fila = 13
desvio = centrado.iloc[fila]

print("valores del empleado:", par.iloc[fila].tolist())
print("desvíos del promedio:", desvio.round(1).tolist())

# TODO: cada desvío por su loading, y sumados
score_a_mano = phi[0] * ___ + phi[1] * ___

scores_par = pca_par.transform(centrado)
score_sklearn = scores_par[fila, 0]

print()
print("score en PC1 calculado a mano:", round(score_a_mano, 2))
print("score en PC1 según sklearn:   ", round(score_sklearn, 2))

Cuánta varianza hay sobre cada dirección está en **`explained_variance_`** (el software lo llama
*autovalor*). Dividido por la varianza total de las dos medidas, da el porcentaje de varianza
explicada. Lo comprobamos por los dos caminos.

In [ ]:
varianza_total = centrado.var().sum()

print("varianza sobre PC1 y PC2:", pca_par.explained_variance_.round(1))
print("varianza total de las dos medidas:", round(varianza_total, 1))
print()
print("PC1, a mano:     ", round(pca_par.explained_variance_[0] / varianza_total, 4))
print("PC1, según sklearn:", round(pca_par.explained_variance_ratio_[0], 4))

El dibujo de todo esto: la nube centrada, **PC1** en verde (la dirección de máxima varianza) y
**PC2** en violeta, perpendicular. El punto naranja es el empleado de recién; el verde, su
proyección sobre PC1.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6), dpi=150)
ax.scatter(centrado.iloc[:, 0], centrado.iloc[:, 1], s=12, color=GRAFITO, alpha=0.3)

largo = 40
ax.plot([-largo * phi[0], largo * phi[0]], [-largo * phi[1], largo * phi[1]], color=VERDE, linewidth=2.5, label="PC1")
phi2 = pca_par.components_[1]
ax.plot([-20 * phi2[0], 20 * phi2[0]], [-20 * phi2[1], 20 * phi2[1]], color=VIOLETA, linewidth=2.5, label="PC2")

ax.scatter(desvio.iloc[0], desvio.iloc[1], s=80, color="#E0823A", zorder=3, label="un empleado")
ax.scatter(score_a_mano * phi[0], score_a_mano * phi[1], s=60, color=VERDE, zorder=3, label="su proyección en PC1")
ax.plot([desvio.iloc[0], score_a_mano * phi[0]], [desvio.iloc[1], score_a_mano * phi[1]], color="#E0823A", linestyle="--")

ax.axhline(0, color="#c5ccd3", linewidth=1)
ax.axvline(0, color="#c5ccd3", linewidth=1)
ax.set_aspect("equal")
ax.set_xlim(-46, 30)
ax.set_ylim(-46, 30)
ax.set_xlabel("objetivos cumplidos (desvío del promedio)")
ax.set_ylabel("entregas a tiempo (desvío del promedio)")
ax.legend(loc="upper left", frameon=False)
for lado in ["top", "right"]:
    ax.spines[lado].set_visible(False)

if os.path.isdir(CARPETA_FIGURAS):
    fig.savefig(CARPETA_FIGURAS + "/clase09_pca2d.png", bbox_inches="tight")
plt.show()

<div class="caja">

**PC1 es la recta que mejor resume la nube**, y hay dos maneras equivalentes de decirlo:

- la dirección en la que las proyecciones (las notas) quedan **más desparramadas**;
- la recta que queda **más cerca de los puntos**, midiendo la distancia en perpendicular.

Son lo mismo por Pitágoras: lo que cada punto no tiene sobre la recta, lo tiene como distancia a
ella, y el total no cambia. No es la recta de regresión, que mide la distancia en vertical porque
predice una variable desde la otra: acá no hay nada que predecir.

</div>

## 3. Las once a la vez

<a id="sec-once"></a>

La misma receta con las once medidas. Una diferencia: como están en unidades distintas, antes de
PCA las **estandarizamos** con `StandardScaler` (a cada una le resta su promedio y la divide por su
desvío). En la sección 4 vemos qué pasa si no lo hacemos.

### ✏️ Consigna 3

Estandarizá las once medidas y ajustá un PCA con **todas** las componentes.

Pistas:

- `StandardScaler().fit_transform(X)` calcula el promedio y el desvío de cada columna y la
  estandariza, todo en un paso;
- `PCA()` sin argumentos se queda con todas las componentes posibles (acá, once);
- ajustalo con la tabla estandarizada, no con `X`.

In [ ]:
from sklearn.preprocessing import StandardScaler

# TODO: estandarizá las once medidas
X_esc = StandardScaler().fit_transform(___)

pca = PCA()
# TODO: ajustá el PCA con la tabla estandarizada
pca.fit(___)

loadings = pd.DataFrame(
    {"PC1": pca.components_[0], "PC2": pca.components_[1]},
    index=MEDIDAS,
)
loadings.round(2).sort_values("PC1", ascending=False)

Leelo columna por columna:

- en **PC1** pesan parecido, alrededor de 0,4, las seis medidas de **cumplimiento** (objetivos,
  líder, entregas, tareas, OKR, foco), y **retrabajos** pesa en negativo: más retrabajos, menos
  nota. Las cuatro de colaboración casi no pesan;
- en **PC2** es al revés: pesan las cuatro de **colaboración** (mentoría, revisiones, pares,
  iniciativas) y el resto casi nada.

Nadie le dijo a PCA que había dos grupos. Los dos bloques de la matriz de correlaciones aparecen
como dos componentes, y el nombre se lo ponemos nosotros: **PC1 = cumplimiento, PC2 =
colaboración**. Otra persona podría llamar a PC1 "productividad": el nombre es una interpretación.

¿Con cuántas componentes nos quedamos? Primero la tabla de varianza explicada, componente por
componente y acumulada.

In [ ]:
varianza = pd.DataFrame({
    "componente": range(1, len(MEDIDAS) + 1),
    "varianza explicada": pca.explained_variance_ratio_,
    "acumulada": pca.explained_variance_ratio_.cumsum(),
})
varianza.round(3)

El **scree plot** es esa tabla dibujada. Se busca el **codo**: el punto desde el cual cada
componente nueva agrega poco, y se retienen las de antes del codo.

In [ ]:
componentes = varianza["componente"]
colores = [VERDE, VIOLETA] + ["#aab4bd"] * (len(MEDIDAS) - 2)

fig, ax = plt.subplots(figsize=(7.5, 4.2), dpi=150)
ax.bar(componentes, varianza["varianza explicada"], color=colores)
ax.plot(componentes, varianza["varianza explicada"], color="#b0572f", marker="o")
ax.axhline(1 / len(MEDIDAS), color=GRAFITO, linestyle="--", linewidth=1, label="lo que aporta una medida sola (1/11)")
ax.set_xticks(componentes)
ax.set_xlabel("componente")
ax.set_ylabel("proporción de varianza explicada")
ax.legend(frameon=False)
for lado in ["top", "right"]:
    ax.spines[lado].set_visible(False)

if os.path.isdir(CARPETA_FIGURAS):
    fig.savefig(CARPETA_FIGURAS + "/clase09_scree.png", bbox_inches="tight")
plt.show()

Las dos primeras explican cerca del 62 % entre las dos, y desde la tercera la línea se aplana: el
codo está en la 3, así que nos quedamos con **dos**. El libro lo admite con todas las letras: el
codo es un criterio a ojo.

Otra regla muy usada es quedarse con las componentes que explican **más que una medida sola** (la
línea punteada, 1/11). Con datos estandarizados eso equivale a una varianza mayor que 1, que es la
**regla de Kaiser**. Acá también da dos:

In [ ]:
print("varianza de cada componente:", pca.explained_variance_.round(2))
print("componentes con varianza mayor que 1:", (pca.explained_variance_ > 1).sum())

El **biplot** junta todo: cada punto es un empleado ubicado según su nota en PC1 y en PC2, y cada
flecha es una medida, dibujada con sus dos loadings. Las flechas que apuntan juntas son medidas que
se mueven juntas.

In [ ]:
scores = pca.transform(X_esc)

fig, ax = plt.subplots(figsize=(7, 6.5), dpi=150)
ax.scatter(scores[:, 0], scores[:, 1], s=10, color=GRAFITO, alpha=0.25)

escala = 4.6 / np.abs(loadings.values).max()
for medida in MEDIDAS:
    x_flecha = loadings.loc[medida, "PC1"] * escala
    y_flecha = loadings.loc[medida, "PC2"] * escala
    if abs(x_flecha) > abs(y_flecha):
        color = VERDE
    else:
        color = VIOLETA
    ax.annotate("", xy=(x_flecha, y_flecha), xytext=(0, 0), arrowprops={"arrowstyle": "->", "color": color, "linewidth": 2})

ax.text(3.9, 0.5, "cumplimiento\n(6 medidas)", color=VERDE, fontsize=9, fontweight="bold")
ax.text(0.4, 4.9, "colaboración (4 medidas)", color=VIOLETA, fontsize=9, fontweight="bold")
ax.text(-4.6, 0.6, "retrabajos", color=VERDE, fontsize=9, fontweight="bold")

ax.axhline(0, color="#c5ccd3", linewidth=1)
ax.axvline(0, color="#c5ccd3", linewidth=1)
ax.set_aspect("equal")
ax.set_xlim(-7, 7)
ax.set_ylim(-6, 7)
ax.set_xlabel("PC1: cumplimiento")
ax.set_ylabel("PC2: colaboración")
for lado in ["top", "right"]:
    ax.spines[lado].set_visible(False)

if os.path.isdir(CARPETA_FIGURAS):
    fig.savefig(CARPETA_FIGURAS + "/clase09_biplot.png", bbox_inches="tight")
plt.show()

Las seis flechas verdes de cumplimiento se pisan porque apuntan casi igual, y por eso no les
pusimos un nombre a cada una: justamente eso queremos ver. Retrabajos apunta al revés, y las cuatro violetas de colaboración van hacia arriba,
a 90 grados: no tienen que ver con las otras.

## 4. ¿Y si no escalamos?

<a id="sec-escalar"></a>

PCA busca la dirección de máxima varianza. Si una medida tiene una varianza enorme **solo por su
unidad**, se queda con PC1. Miremos la varianza de cada medida en sus unidades originales.

In [ ]:
X.var().round(2).sort_values(ascending=False)

Minutos de mentoría tiene una varianza de casi 7.000; la siguiente, alrededor de 150. Hacemos PCA
solo centrando, sin estandarizar:

In [ ]:
pca_crudo = PCA()
pca_crudo.fit(X - X.mean())

loadings_crudo = pd.Series(pca_crudo.components_[0], index=MEDIDAS)

print("PC1 sin escalar explica:", round(pca_crudo.explained_variance_ratio_[0], 3))
print()
print(loadings_crudo.abs().round(3).sort_values(ascending=False))

Sin escalar, PC1 es **minutos de mentoría y nada más**: explica casi el 95 %, y no porque la
mentoría sea lo más importante del desempeño, sino porque está medida en minutos. Si la midiéramos
en horas, desaparecería. Salvo que todas las medidas estén en la misma unidad, **se escala siempre**.

Hay más de una forma de escalar. `StandardScaler` hace el **z-score**; `MinMaxScaler` lleva cada
medida al rango de 0 a 1. Para este análisis dan la misma estructura:

In [ ]:
from sklearn.preprocessing import MinMaxScaler

pca_minmax = PCA()
pca_minmax.fit(MinMaxScaler().fit_transform(X))

print("z-score,  varianza de las 3 primeras:", pca.explained_variance_ratio_[:3].round(3))
print("min-max,  varianza de las 3 primeras:", pca_minmax.explained_variance_ratio_[:3].round(3))

## 5. PC1 y la renuncia

<a id="sec-renuncia"></a>

PCA no miró la renuncia en ningún momento. Pero ya tenemos una nota de cumplimiento para cada
empleado: ¿sirve para predecir quién se va?

In [ ]:
datos["renuncio"] = datos["renuncia"] == "Si"
datos["pc1"] = scores[:, 0]

datos.groupby("renuncio")["pc1"].mean().round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8), dpi=150)
ax.hist(datos.loc[~datos["renuncio"], "pc1"], bins=30, density=True, color=GRAFITO, alpha=0.5, label="se quedaron")
ax.hist(datos.loc[datos["renuncio"], "pc1"], bins=30, density=True, color="#d6604d", alpha=0.5, label="renunciaron")
ax.set_xlabel("nota en PC1 (cumplimiento)")
ax.set_ylabel("proporción")
ax.legend(frameon=False)
for lado in ["top", "right"]:
    ax.spines[lado].set_visible(False)

if os.path.isdir(CARPETA_FIGURAS):
    fig.savefig(CARPETA_FIGURAS + "/clase09_pc1_renuncia.png", bbox_inches="tight")
plt.show()

Los que renunciaron tienen, en promedio, bastante menos cumplimiento. Lo medimos con la **AUC**,
con validación cruzada repetida como en la clase 8, comparando tres modelos de regresión
logística:

- con las **once medidas**, estandarizadas;
- con **una sola columna**: la nota en PC1;
- con PC1 pero **sin escalar**.

Cada modelo es un **`Pipeline`**: una lista de pasos que se ajustan juntos. En la sección 6 vemos
por qué eso importa.

### ✏️ Consigna 4

Completá los tres modelos y medí su AUC con la validación cruzada `CV`.

Pistas:

- `make_pipeline(paso1, paso2, ...)` arma la lista de pasos en orden: primero lo que prepara los
  datos, al final el modelo;
- el de "solo PC1" lleva tres pasos: estandarizar, PCA con **una** componente y la logística;
- el de "sin escalar" es igual, pero sin el primer paso;
- `cross_val_score(modelo, X, y, cv=CV, scoring="roc_auc")` devuelve una AUC por cada fold, y
  `.mean()` las promedia.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

y = datos["renuncio"]
CV = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEED)

modelo_once = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
# TODO: estandarizar, quedarse con UNA componente, y la logística
modelo_pc1 = make_pipeline(___, PCA(n_components=___), LogisticRegression(max_iter=2000))
# TODO: lo mismo pero SIN estandarizar
modelo_crudo = make_pipeline(___, LogisticRegression(max_iter=2000))

auc_once = cross_val_score(modelo_once, X, y, cv=CV, scoring="roc_auc").mean()
# TODO: la misma validación cruzada para los otros dos
auc_pc1 = cross_val_score(modelo_pc1, X, y, cv=___, scoring="roc_auc").mean()
auc_crudo = cross_val_score(modelo_crudo, X, y, cv=___, scoring="roc_auc").mean()

pd.Series(
    [auc_once, auc_pc1, auc_crudo],
    index=["las 11 medidas", "solo PC1", "PC1 sin escalar"],
    name="AUC",
).round(3)

Con **una columna** la AUC es la misma que con once: comprimimos once medidas en una sin perder
poder predictivo. Sin escalar, PC1 pasa a ser minutos de mentoría y la AUC cae a la de tirar una
moneda (0,5).

<div class="ojo">

**Dos cuidados.**

- Que PC1 prediga la renuncia es una propiedad de **estos datos**, no de PCA. PCA no mira la
  renuncia: busca la dirección de más varianza, y acá coincide con la que cae antes de irse
  porque Nimbus se armó así. En otros datos la componente que más varía puede no tener nada que ver
  con lo que querés predecir.
- Se pierde **interpretabilidad**. El coeficiente de PC1 dice "más cumplimiento, menos renuncia",
  pero ya no puede decir cuál de las once medidas es la que importa. Si la pregunta es
  explicativa, eso pesa.

</div>

## 6. Leakage: el experimento del ruido

<a id="sec-leakage"></a>

Escalar, PCA y elegir variables **aprenden algo de los datos** (la media y el desvío, los loadings,
cuáles variables sirven). Si lo aprenden con todos los datos antes de partir en entrenamiento y
testeo, el testeo ya ayudó a armar el modelo: eso es **leakage**, fuga de información.

Con el escalado en Nimbus el efecto es chico. Para ver cuánto puede engañar, repetimos un
experimento clásico del libro de Hastie, Tibshirani y Friedman, con **datos simulados**:

1. 50 personas y 5.000 variables sacadas al azar;
2. una respuesta también al azar, mitad sí y mitad no: por construcción, nada la puede predecir;
3. un paso de preprocesado: elegir las 100 variables que más se relacionan con la respuesta;
4. una logística evaluada con validación cruzada.

Lo hacemos de dos maneras, en 20 datasets simulados distintos: eligiendo las 100 variables **con
todos los datos** y después validando, y eligiéndolas **adentro de cada fold**, con un `Pipeline`.

### ✏️ Consigna 5

La parte "mal" del experimento ya está escrita. Completá la parte "bien": la elección de las 100
variables tiene que ser un paso del `Pipeline`, para que se rehaga adentro de cada fold.

Pistas:

- el paso de elegir es `SelectKBest(f_classif, k=100)`, sin `.fit()`: lo ajusta
  `cross_val_score`, fold por fold;
- a `cross_val_score` se le pasa el `Pipeline` entero y **todas** las 5.000 variables
  (`X_ruido`), no las ya elegidas.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

auc_afuera = []
auc_adentro = []

for semilla in range(20):
    rng = np.random.default_rng(semilla)
    X_ruido = rng.normal(size=(50, 5000))
    y_ruido = np.repeat([0, 1], 25)
    cv_ruido = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=semilla)

    # Mal: se eligen las 100 variables mirando a las 50 personas, y después se valida.
    elegidas = SelectKBest(f_classif, k=100).fit(X_ruido, y_ruido).get_support()
    mal = cross_val_score(LogisticRegression(max_iter=2000), X_ruido[:, elegidas], y_ruido, cv=cv_ruido, scoring="roc_auc")
    auc_afuera.append(mal.mean())

    # TODO: la elección de las 100 variables como primer paso del Pipeline
    modelo = make_pipeline(___, LogisticRegression(max_iter=2000))
    # TODO: el Pipeline entero, con TODAS las variables
    bien = cross_val_score(___, ___, y_ruido, cv=cv_ruido, scoring="roc_auc")
    auc_adentro.append(bien.mean())

print("eligiendo con todos los datos, AUC promedio:", round(np.mean(auc_afuera), 3))
print("eligiendo adentro de cada fold, AUC promedio:", round(np.mean(auc_adentro), 3))

Con ruido puro, elegir las variables con todos los datos da una AUC **perfecta**. Es falsa: entre
5.000 variables al azar siempre hay algunas que coinciden por casualidad con la respuesta de esas
50 personas, y como las elegimos mirando a todas, también miramos a las que después hacen de
testeo. Cuando la elección se hace adentro de cada fold, las coincidencias casuales del
entrenamiento no se repiten en gente nueva, y la AUC da lo que tiene que dar: alrededor de 0,5.

La regla: **todo lo que aprende algo de los datos va adentro del `Pipeline`**, y el `Pipeline`
entero va adentro de la validación cruzada. `cross_val_score` se encarga de ajustar cada paso solo
con el entrenamiento de cada fold.

## 7. ¿Cuántas componentes? Que decida la validación cruzada

<a id="sec-cuantas"></a>

El codo del scree plot es un criterio a ojo. Si el objetivo es **predecir**, hay una forma mejor:
probar. Como la cantidad de componentes es un paso del `Pipeline`, se busca con `GridSearchCV`,
igual que buscábamos la profundidad de un árbol en la clase 8. El nombre del parámetro es el del
paso (`pca`), dos guiones bajos, y el del argumento (`n_components`).

### ✏️ Consigna 6

Buscá con `GridSearchCV` cuántas componentes conviene retener, probando de 1 a 11.

Pistas:

- la grilla es un diccionario: la clave es el nombre del hiperparámetro y el valor, la lista de
  candidatos;
- adentro de un `Pipeline` el nombre es el del paso, dos guiones bajos, y el del argumento:
  `"pca__n_components"`;
- `list(range(1, 12))` es la lista del 1 al 11.

In [ ]:
from sklearn.model_selection import GridSearchCV

busqueda = GridSearchCV(
    modelo_pc1,
    # TODO: el nombre del hiperparámetro y la lista de candidatos, de 1 a 11
    {"___": list(range(1, ___))},
    cv=CV,
    scoring="roc_auc",
)
busqueda.fit(X, y)

resultados = pd.DataFrame(busqueda.cv_results_)
resultados[["param_pca__n_components", "mean_test_score"]].round(3)

Con **una sola componente** ya estamos en la mejor AUC, y agregar más no mejora: una meseta. Para
predecir la renuncia alcanza con la nota de cumplimiento.

## 8. Más allá de lo lineal

<a id="sec-nolineal"></a>

PCA solo encuentra **direcciones rectas**. Nimbus no tiene otra forma de estructura, así que para
verlo usamos un clásico que viene con scikit-learn: 1.797 imágenes de **dígitos escritos a mano**,
de 8 × 8 píxeles. Cada imagen es una fila de 64 números (cuánta tinta hay en cada píxel), así que
es una tabla como la de Nimbus, con 64 columnas en lugar de 11.

Comparamos PCA con **t-SNE**, un método que intenta que los puntos cercanos en las 64 dimensiones
sigan cercanos en el dibujo. Los 64 píxeles están en la misma unidad, así que acá **no** se
estandariza. Este paso tarda unos segundos.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE

digitos = load_digits()

en_pca = PCA(n_components=2).fit_transform(digitos.data)
en_tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED).fit_transform(digitos.data)

fig, ejes = plt.subplots(1, 2, figsize=(11, 5), dpi=150)
ejes[0].scatter(en_pca[:, 0], en_pca[:, 1], c=digitos.target, cmap="tab10", s=6)
ejes[0].set_title("PCA")
puntos = ejes[1].scatter(en_tsne[:, 0], en_tsne[:, 1], c=digitos.target, cmap="tab10", s=6)
ejes[1].set_title("t-SNE")
for eje in ejes:
    eje.set_xticks([])
    eje.set_yticks([])
fig.colorbar(puntos, ax=ejes, ticks=range(10), label="dígito")

if os.path.isdir(CARPETA_FIGURAS):
    fig.savefig(CARPETA_FIGURAS + "/clase09_tsne.png", bbox_inches="tight")
plt.show()

Con PCA los diez dígitos se pisan; con t-SNE aparecen separados. Hay estructura, pero no es recta.

Cada método sirve para otra cosa:

| | PCA | t-SNE |
|---|---|---|
| qué busca | direcciones rectas de máxima varianza | que los vecinos sigan siendo vecinos |
| se puede leer | sí: los loadings dicen qué pesa | no: los ejes no significan nada |
| da siempre lo mismo | sí | no: cambia con la semilla y con `perplexity` |
| se aplica a casos nuevos | sí, con `.transform()` | no |
| sirve para | resumir, armar índices, preparar datos para un modelo | mirar si hay grupos |

En t-SNE tampoco significan nada las **distancias entre grupos** ni su tamaño. **UMAP** es una
alternativa más rápida que sí se aplica a casos nuevos; no viene con scikit-learn (se instala con
`pip install umap-learn`). La regla: con t-SNE y UMAP se **descubre**, con PCA se **mide**.

## 9. Ejercicio de cierre: la encuesta de clima

<a id="sec-clima"></a>

En la clase 5 usamos la **encuesta de clima 2026** (`nimbus_clima.csv`): el índice de bienestar
de cada empleado y 18 medidas más (horas extra, apoyo del equipo, reuniones, días de home
office, cursos, etcétera). ¿Se comprimen tan bien como las once medidas de desempeño?

### ✏️ Consigna 7

Hacé un PCA sobre las 18 medidas de la encuesta de clima, **sin** el índice de bienestar, y
mirá cuánta varianza explica cada componente.

Pistas:

- `clima.drop(columns=[...])` devuelve la tabla sin esas columnas: sacá `"empleado_id"` y
  `"bienestar_laboral"`;
- son medidas en unidades distintas: estandarizá antes, igual que en la sección 3;
- la varianza de cada componente está en `.explained_variance_`: comparala con 1.

In [ ]:
clima = pd.read_csv(BASE + "nimbus_clima.csv")
# TODO: sacá la columna del empleado y la del índice de bienestar; quedan las medidas
predictores = clima.drop(columns=[___, ___])

pca_clima = PCA()
# TODO: estandarizá las medidas y ajustá el PCA, como en la sección 3
pca_clima.fit(___)

varianza_clima = pd.DataFrame({
    "componente": range(1, predictores.shape[1] + 1),
    "varianza explicada": pca_clima.explained_variance_ratio_,
    "acumulada": pca_clima.explained_variance_ratio_.cumsum(),
})

print(predictores.shape[1], "medidas")
# TODO: ¿cuántas componentes tienen varianza mayor que 1?
print("componentes con varianza mayor que 1:", ___)
varianza_clima.round(3).head(8)

Ahora las dos curvas juntas, desempeño y clima, y la correlación más fuerte que hay entre dos
medidas de cada tabla.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4), dpi=150)
ax.plot(varianza["componente"], varianza["varianza explicada"], color=VERDE, marker="o", label="desempeño (11 medidas)")
ax.plot(varianza_clima["componente"], varianza_clima["varianza explicada"], color=GRAFITO, marker="o", label="clima (18 medidas)")
ax.set_xticks(varianza_clima["componente"])
ax.set_xlabel("componente")
ax.set_ylabel("proporción de varianza explicada")
ax.legend(frameon=False)
for lado in ["top", "right"]:
    ax.spines[lado].set_visible(False)

if os.path.isdir(CARPETA_FIGURAS):
    fig.savefig(CARPETA_FIGURAS + "/clase09_scree_clima.png", bbox_inches="tight")
plt.show()

valores_desempeno = np.abs(X.corr().values)
np.fill_diagonal(valores_desempeno, 0)
valores_clima = np.abs(predictores.corr().values)
np.fill_diagonal(valores_clima, 0)

print("correlación más fuerte entre dos medidas de desempeño:", round(valores_desempeno.max(), 2))
print("correlación más fuerte entre dos medidas de clima:    ", round(valores_clima.max(), 2))

En clima **no hay codo**: la curva baja de a poco, y las dos primeras componentes juntan apenas
16 % de la varianza, contra 62 % en desempeño. Ninguna componente resume mucho más que una
medida sola.

La razón está en la última línea: en desempeño hay medidas que se repiten entre sí, y en clima
cada una dice algo bastante propio. **PCA comprime redundancia**: si no la hay, no tiene qué
comprimir.

Mirá también la regla de Kaiser: en clima deja **ocho** componentes, cada una apenas por encima
de 1. Cuando no hay estructura, una regla automática igual devuelve un número; por eso no se usa
sola, sin mirar el scree plot.

## 10. Cierre

<a id="sec-cierre"></a>

### Lo que te llevás

1. **Reducir dimensiones es encontrar direcciones.** Muchas medidas que se mueven juntas se
   resumen en pocas componentes; el nombre se lo ponemos nosotros, y a cambio se pierde la lectura
   directa de cada medida.
2. **PC1 es la dirección de máxima varianza**, que es la misma recta que queda más cerca de los
   puntos. PC2, la máxima varianza que queda, a 90 grados.
3. **Loadings, scores y biplot.** Los loadings son la receta; el score, la nota de cada caso; el
   biplot, las dos cosas en un dibujo.
4. **Cuántas componentes es una decisión.** Para describir, el codo y la regla de Kaiser; para
   predecir, validación cruzada.
5. **Escalar antes**, salvo que todo esté en la misma unidad.
6. **Todo lo que aprende de los datos va adentro del `Pipeline`.**

### Todo el código de hoy, en pocas líneas

```python
modelo = make_pipeline(StandardScaler(), PCA(n_components=1), LogisticRegression(max_iter=2000))
grupos = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=49)
auc = cross_val_score(modelo, X, y, cv=grupos, scoring="roc_auc")
print(auc.mean())
```

### Para seguir

- James y otros, capítulo 12, secciones 1 y 2, es la lectura de esta clase; la sección 6.3.1
  cuenta cómo usar componentes para predecir.
- La clase que viene: **clustering**. ¿Hay grupos de empleados en el plano de cumplimiento y
  colaboración?